# 07 - Policy Monitoring & Efficacy Analysis

**Purpose**: Monitor retention policies and measure their impact on churn (per spec US6).

**Spec Reference**: `specs/001-churn-prediction-model/spec.md`

## Analysis Goals
- Compare churn rates between policy-receiving and control groups
- Statistical significance testing (chi-square, lift analysis)
- Track policy performance over time
- Generate efficacy reports for stakeholders

## Constitution Alignment
- **Principle V (Continuous Learning)**: Measure policy impact for iteration
- **Principle IV (Actionable Insights)**: Guide policy improvements

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats

# Project imports
import sys
sys.path.append('..')
from src.utils.config import get_config
from src.utils.logging import setup_logging, get_logger
from src.reporting.policy_efficacy import (
    compare_cohorts,
    calculate_lift,
    chi_square_test,
    create_efficacy_report
)

# Setup
setup_logging()
logger = get_logger(__name__)
config = get_config()

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d')}")

## 1. Load Policy and Outcome Data

⚠️ **ACTION REQUIRED**: You need a `policy_tracking` table with:
- customer_id
- policy_name (e.g., 'discount_offer', 'outreach_call')
- policy_start_date
- policy_end_date

In [ ]:
# Load data from Lakehouse
# policy_df = spark.read.format("delta").load("Tables/policy_tracking").toPandas()
# predictions_df = spark.read.format("delta").load("Tables/predictions").toPandas()
# churn_events_df = spark.read.format("delta").load("Tables/churn_events").toPandas()

# print(f"Loaded {len(policy_df)} policy records")
# print(f"Loaded {len(predictions_df)} predictions")
# print(f"Loaded {len(churn_events_df)} churn events")

## 2. Define Analysis Parameters

In [ ]:
# Analysis parameters
ANALYSIS_PERIOD_DAYS = 90  # Look at outcomes over 90 days
MIN_COHORT_SIZE = 30       # Minimum customers for valid comparison
SIGNIFICANCE_LEVEL = 0.05  # p-value threshold

# Define which policies to analyze
POLICIES_TO_ANALYZE = [
    'retention_discount',
    'account_manager_outreach',
    'product_training',
    'loyalty_program'
]

## 3. Create Treatment and Control Groups

In [ ]:
# Define treatment vs control cohorts
# For each policy, treatment = received policy, control = similar risk but no policy

# def create_matched_cohorts(policy_df, predictions_df, policy_name):
#     """
#     Create matched treatment and control groups based on risk score.
#     """
#     # Treatment: customers who received the policy
#     treatment_ids = policy_df[policy_df['policy_name'] == policy_name]['customer_id'].unique()
#     
#     # Get risk scores at policy start
#     treatment_risks = predictions_df[predictions_df['customer_id'].isin(treatment_ids)]
#     
#     # Control: customers with similar risk but no policy
#     all_policy_ids = policy_df['customer_id'].unique()
#     control_candidates = predictions_df[~predictions_df['customer_id'].isin(all_policy_ids)]
#     
#     # Match on risk tier
#     treatment_risk_dist = treatment_risks.groupby('risk_tier').size()
#     control_matched = control_candidates.groupby('risk_tier').apply(
#         lambda x: x.sample(min(len(x), treatment_risk_dist.get(x.name, 0)), random_state=42)
#     ).reset_index(drop=True)
#     
#     return treatment_ids, control_matched['customer_id'].unique()

# # Example: Create cohorts for first policy
# # treatment_ids, control_ids = create_matched_cohorts(policy_df, predictions_df, POLICIES_TO_ANALYZE[0])

## 4. Calculate Policy Efficacy

In [ ]:
# Analyze each policy
# efficacy_results = []

# for policy_name in POLICIES_TO_ANALYZE:
#     print(f"\nAnalyzing: {policy_name}")
#     print("-" * 50)
#     
#     # Create cohorts
#     treatment_ids, control_ids = create_matched_cohorts(policy_df, predictions_df, policy_name)
#     
#     if len(treatment_ids) < MIN_COHORT_SIZE:
#         print(f"  ⚠️ Insufficient treatment cohort size ({len(treatment_ids)} < {MIN_COHORT_SIZE})")
#         continue
#     
#     # Calculate churn outcomes
#     treatment_churned = len(churn_events_df[churn_events_df['customer_id'].isin(treatment_ids)])
#     control_churned = len(churn_events_df[churn_events_df['customer_id'].isin(control_ids)])
#     
#     treatment_rate = treatment_churned / len(treatment_ids)
#     control_rate = control_churned / len(control_ids) if len(control_ids) > 0 else 0
#     
#     # Calculate lift and significance
#     lift = calculate_lift(control_rate, treatment_rate)
#     
#     chi2, p_value = chi_square_test(
#         treatment_total=len(treatment_ids),
#         treatment_churned=treatment_churned,
#         control_total=len(control_ids),
#         control_churned=control_churned
#     )
#     
#     result = {
#         'policy_name': policy_name,
#         'treatment_size': len(treatment_ids),
#         'control_size': len(control_ids),
#         'treatment_churn_rate': treatment_rate,
#         'control_churn_rate': control_rate,
#         'churn_reduction': control_rate - treatment_rate,
#         'lift_pct': lift * 100,
#         'chi2_statistic': chi2,
#         'p_value': p_value,
#         'is_significant': p_value < SIGNIFICANCE_LEVEL
#     }
#     efficacy_results.append(result)
#     
#     print(f"  Treatment cohort: {len(treatment_ids)} customers")
#     print(f"  Control cohort: {len(control_ids)} customers")
#     print(f"  Treatment churn rate: {treatment_rate:.2%}")
#     print(f"  Control churn rate: {control_rate:.2%}")
#     print(f"  Lift: {lift:+.1%}")
#     print(f"  p-value: {p_value:.4f}")
#     print(f"  Significant: {'✓ YES' if p_value < SIGNIFICANCE_LEVEL else '✗ NO'}")

# efficacy_df = pd.DataFrame(efficacy_results)

## 5. Visualize Policy Performance

In [ ]:
# Bar chart comparing treatment vs control
# if not efficacy_df.empty:
#     fig, ax = plt.subplots(figsize=(12, 6))
#     
#     x = np.arange(len(efficacy_df))
#     width = 0.35
#     
#     bars1 = ax.bar(x - width/2, efficacy_df['control_churn_rate'], width, 
#                    label='Control', color='lightcoral')
#     bars2 = ax.bar(x + width/2, efficacy_df['treatment_churn_rate'], width,
#                    label='Treatment (Received Policy)', color='lightgreen')
#     
#     # Add significance markers
#     for i, row in efficacy_df.iterrows():
#         if row['is_significant']:
#             ax.annotate('*', xy=(i, max(row['control_churn_rate'], row['treatment_churn_rate']) + 0.02),
#                        ha='center', fontsize=20)
#     
#     ax.set_ylabel('Churn Rate')
#     ax.set_title('Policy Efficacy: Churn Rate by Treatment Group\n(* = statistically significant)')
#     ax.set_xticks(x)
#     ax.set_xticklabels(efficacy_df['policy_name'], rotation=45, ha='right')
#     ax.legend()
#     ax.set_ylim(0, efficacy_df[['control_churn_rate', 'treatment_churn_rate']].max().max() * 1.3)
#     
#     plt.tight_layout()
#     plt.show()

In [ ]:
# Lift visualization
# if not efficacy_df.empty:
#     fig, ax = plt.subplots(figsize=(10, 6))
#     
#     colors = ['green' if x > 0 else 'red' for x in efficacy_df['lift_pct']]
#     bars = ax.barh(efficacy_df['policy_name'], efficacy_df['lift_pct'], color=colors)
#     
#     ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
#     ax.set_xlabel('Lift (% reduction in churn)')
#     ax.set_title('Policy Efficacy: Churn Reduction Lift')
#     
#     # Add significance indicators
#     for i, (bar, sig) in enumerate(zip(bars, efficacy_df['is_significant'])):
#         if sig:
#             ax.annotate('p<0.05', xy=(bar.get_width() + 1, bar.get_y() + bar.get_height()/2),
#                        va='center', fontsize=8, color='blue')
#     
#     plt.tight_layout()
#     plt.show()

## 6. Time-Series Analysis

Track policy performance over time to detect trends.

In [ ]:
# Monthly efficacy trends
# This requires policy_df to have date information

# def calculate_monthly_efficacy(policy_df, churn_events_df, policy_name):
#     policy_df['month'] = pd.to_datetime(policy_df['policy_start_date']).dt.to_period('M')
#     
#     monthly_results = []
#     for month in policy_df['month'].unique():
#         month_policies = policy_df[(policy_df['policy_name'] == policy_name) & 
#                                    (policy_df['month'] == month)]
#         if len(month_policies) < 10:
#             continue
#             
#         treatment_ids = month_policies['customer_id'].values
#         churned = len(churn_events_df[churn_events_df['customer_id'].isin(treatment_ids)])
#         
#         monthly_results.append({
#             'month': month,
#             'churn_rate': churned / len(treatment_ids),
#             'cohort_size': len(treatment_ids)
#         })
#     
#     return pd.DataFrame(monthly_results)

# # Plot trends
# # monthly_efficacy = calculate_monthly_efficacy(policy_df, churn_events_df, 'retention_discount')
# # monthly_efficacy.plot(x='month', y='churn_rate', kind='line', marker='o')

## 7. Generate Efficacy Report

In [ ]:
# Generate summary report
# if not efficacy_df.empty:
#     report = create_efficacy_report(efficacy_df)
#     print(report)

## 8. Save Results to Lakehouse

In [ ]:
# Prepare policy efficacy records
# import uuid

# efficacy_records = []
# for _, row in efficacy_df.iterrows():
#     record = {
#         'efficacy_id': str(uuid.uuid4()),
#         'policy_name': row['policy_name'],
#         'analysis_date': datetime.now().date(),
#         'analysis_period_days': ANALYSIS_PERIOD_DAYS,
#         'treatment_size': row['treatment_size'],
#         'control_size': row['control_size'],
#         'treatment_churn_rate': row['treatment_churn_rate'],
#         'control_churn_rate': row['control_churn_rate'],
#         'churn_reduction': row['churn_reduction'],
#         'lift_pct': row['lift_pct'],
#         'chi2_statistic': row['chi2_statistic'],
#         'p_value': row['p_value'],
#         'is_significant': row['is_significant'],
#         'created_at': datetime.now()
#     }
#     efficacy_records.append(record)

# efficacy_output = pd.DataFrame(efficacy_records)

In [ ]:
# Save to Lakehouse
# spark_df = spark.createDataFrame(efficacy_output)
# spark_df.write.format("delta").mode("append").save("Tables/policy_efficacy")
# print("Policy efficacy results saved to Lakehouse!")

## Summary & Recommendations

### Policy Performance Summary

| Policy | Treatment Rate | Control Rate | Lift | Significant |
|--------|----------------|--------------|------|-------------|
| [Policy 1] | X% | X% | +/-X% | Yes/No |
| [Policy 2] | X% | X% | +/-X% | Yes/No |

### Recommendations

1. **Scale Up**: [Policies with significant positive lift]
2. **Iterate**: [Policies with positive but non-significant lift]
3. **Discontinue**: [Policies with negative or no lift]
4. **Test**: [New policy ideas based on feature importance]

### Next Steps
- Share findings with business stakeholders
- Design A/B tests for new policy variants
- Re-run analysis after 30/60/90 days